# KVO_08 — Satellite prediction of Musanga canopy fraction

**Five sites:** MOS_1_2, Yoko, Yangambi, Baego, Luki.

**Phase 1: data-preparation/validation.** The first runnable cells audit existing NB04 products without changing them. The NB04 production manifest documents the four-site final frozen model, uint16 0–10000 probability and 65535 NoData. Luki is an external site. The four current NB04 maps may be annotation-ROI restricted; their *pre-vector-ROI-mask* backups have broader coverage but must not be assumed full mosaic until audited.

**CRS rule:** keep UAV source pixels in their native projected CRS (Luki EPSG:32733; other sites verified individually), and later construct per-site target grids aligned to a chosen **actual Sentinel-2 10 m projection + transform**. Satellite embeddings and S2 are *not assumed* co-aligned; verify exact grid transforms first. No reprojection is performed by the audit.

**Run Section 00 and new Section 04 first (CPU), then review earlier audits only if needed.** They only read metadata and small samples of probability maps; they never run U-Net, change any raster, or download satellites. All results go to `KVO_work/KVO_08_satellite_fraction/validation/`.

## 00 — Paths, site inventory and processing contract

**CPU / read-only.** Define native UAV sources, the current four-site/Luki outputs and optional full-coverage candidates. No source rasters are modified.

In [ ]:
# BLOCK 00 — canonical site paths; CPU-only, read-only inspection
from pathlib import Path
import json, math, shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from IPython.display import display
from google.colab import drive

MY = Path("/content/drive/MyDrive")
if not MY.exists(): drive.mount("/content/drive")
ROOT = MY / "Colab Notebooks" / "Musanga"
GEO = ROOT / "data" / "Geo_Ref"
KVO04 = ROOT / "KVO_work" / "KVO_04_final_multisite"
MAPS = KVO04 / "probability_maps"
OUT = ROOT / "KVO_work" / "KVO_08_satellite_fraction"
VALIDATION = OUT / "validation"
VALIDATION.mkdir(parents=True, exist_ok=True)

RGB = {
    "MOS_1_2": GEO / "MOS_1_2_kvo.tif",
    "Yoko": GEO / "Yoko_kvo_georef.tif",
    "Yangambi": GEO / "Yangambi_georef.tif",
    "Baego": GEO / "Baego_georef.tif",
    "Luki": GEO / "Luki.tif",
}
CURRENT = {site: MAPS / f"{site}_musanga_probability_uint16.tif" for site in RGB}
BACKUP = {
    site: MAPS / f"{site}_musanga_probability_uint16.pre_vector_roi_mask.tif"
    for site in RGB if site != "Luki"
}
FULL_LUKI = MAPS / "Luki_musanga_probability_uint16_FULL.tif"

NODATA = 65535
SCALE = 10000
SITES = list(RGB)
print("Validation outputs:", VALIDATION)
print("Sites:", ", ".join(SITES))
print("Luki FULL map present:", FULL_LUKI.exists())
print("No inference, overwrites, reprojecting, or downloads in Blocks 00–02.")

## 01 — Raster metadata, CRS and probability schema

**CPU / read-only.** Verify raster dimensions, pixel origins, CRS, probability encoding and NoData. Current UAV mosaics are EPSG:32635 for MOS, Yoko, Yangambi and Baego; Luki is EPSG:32733. These are *not* reprojected in this step.

In [ ]:
# BLOCK 01 — Native-CRS metadata and probability schema checks (READ-ONLY)
# Candidate types are recorded explicitly. 'pre_ROI_backup' is NOT assumed full coverage.
import warnings
def meta(path):
    with rasterio.open(path) as ds:
        return {
            "width": ds.width, "height": ds.height, "crs": str(ds.crs),
            "transform": tuple(ds.transform), "bounds": tuple(ds.bounds),
            "px_x_m": abs(ds.transform.a), "px_y_m": abs(ds.transform.e),
            "dtype": ds.dtypes[0], "bands": ds.count, "nodata": ds.nodata,
            "tags": ds.tags(),
            "bbox_ha": ds.width*ds.height*abs(
                ds.transform.a*ds.transform.e-ds.transform.b*ds.transform.d
            )/10000,
        }

rows = []
candidates = []
for site in SITES:
    rgb_path = RGB[site]
    base = {"site":site,"rgb_path":str(rgb_path),
            "rgb_exists":rgb_path.is_file(),
            "current_exists":CURRENT[site].is_file()}
    if not rgb_path.is_file():
        rows.append({**base,"status":"MISSING_RGB"})
        continue
    rgb_meta = meta(rgb_path)
    assert rgb_meta["crs"]!="None",f"{site}: RGB CRS is undefined"
    base.update(rgb_crs=rgb_meta["crs"],
                gsd_x_m=rgb_meta["px_x_m"],gsd_y_m=rgb_meta["px_y_m"],
                rgb_bands=rgb_meta["bands"],
                width=rgb_meta["width"],height=rgb_meta["height"],
                rgb_bbox_ha=rgb_meta["bbox_ha"])
    candidate_paths = [("current",CURRENT[site])]
    if site in BACKUP: candidate_paths.append(("pre_ROI_backup",BACKUP[site]))
    if site=="Luki": candidate_paths.append(("full_mosaic",FULL_LUKI))
    for kind,p in candidate_paths:
        item={**base,"candidate_kind":kind,"probability_path":str(p),
              "probability_exists":p.is_file()}
        if not p.is_file():
            item["status"]="MISSING_PROBABILITY"
            rows.append(item)
            continue
        pm=meta(p)
        same_grid=(pm["width"]==rgb_meta["width"] and
                   pm["height"]==rgb_meta["height"] and
                   pm["crs"]==rgb_meta["crs"] and
                   np.allclose(pm["transform"],rgb_meta["transform"],rtol=0,atol=1e-6))
        schema_ok=(pm["bands"]==1 and pm["dtype"]=="uint16" and
                   pm["nodata"]==NODATA)
        item.update(prob_crs=pm["crs"],prob_dtype=pm["dtype"],
                    prob_nodata=pm["nodata"],same_uav_grid=same_grid,
                    schema_ok=schema_ok,coverage_tag=pm["tags"].get("COVERAGE",""),
                    model_checkpoint=pm["tags"].get("MODEL_CHECKPOINT",""),
                    probability_file_GB=round(p.stat().st_size/1e9,3),
                    status="SCHEMA_OK" if same_grid and schema_ok else "SCHEMA_ERROR")
        rows.append(item)
        candidates.append((site,kind,p,item["status"]))
audit=pd.DataFrame(rows)
audit.to_csv(VALIDATION/"01_reference_metadata.csv",index=False)
display(audit[["site","candidate_kind","probability_exists","rgb_crs",
               "rgb_bbox_ha","same_uav_grid","schema_ok","coverage_tag",
               "status"]].round(2))
assert any(z[3]=="SCHEMA_OK" for z in candidates),"No valid probability candidate found."
print("IMPORTANT: current and backup files have identical raster *dimensions*;")
print("this says nothing about their actual valid-pixel coverage.")

## 02 — Sample valid-data coverage and candidate assessment

**CPU / read-only.** Sample the same image locations for available probability products. A missing backup must be reported as missing, not recreated or silently substituted. Sampling is a screening diagnostic, **not** a definitive full-mosaic coverage measurement.

**Observed run, 11 October 2026:** current maps exist for all five sites and have correct native-grid metadata; the four pre-ROI backup files were not found at the expected names, and the independent full-Luki mosaic is still pending. Proceed with the audit, then run NB04 Block 09 for full Luki inference. Validate whether the four other maps have the desired whole-mosaic coverage before generating satellite target fractions.

In [ ]:
# BLOCK 02 — Stratified same-location probability coverage comparison
# Fast read-only sample; NOT an exact entire-mosaic valid-area computation.
# Sample 12 x 12 source-coordinate locations with 128-px windows each.
GRID_N = 12
SAMPLE_PX = 128
sample_rows=[]
for site in SITES:
    if not RGB[site].exists():continue
    site_candidates=[(kind,path) for s,kind,path,status in candidates
                     if s==site and status=="SCHEMA_OK"]
    if not site_candidates:continue
    with rasterio.open(RGB[site]) as src:
        locations=[]
        for yf in (np.arange(GRID_N)+.5)/GRID_N:
            for xf in (np.arange(GRID_N)+.5)/GRID_N:
                row=int(yf*src.height);col=int(xf*src.width)
                row=max(0,min(row-SAMPLE_PX//2,src.height-SAMPLE_PX))
                col=max(0,min(col-SAMPLE_PX//2,src.width-SAMPLE_PX))
                locations.append(Window(col,row,min(SAMPLE_PX,src.width),min(SAMPLE_PX,src.height)))
        area_ha=src.width*src.height*abs(
            src.transform.a*src.transform.e-src.transform.b*src.transform.d)/1e4
    for kind,path in site_candidates:
        n=0;valid_total=0;invalid_encoding=0;mean_sum=0.0
        counts=[]
        with rasterio.open(path) as ds:
            for w in locations:
                a=ds.read(1,window=w)
                good=(a!=NODATA)&(a<=SCALE)
                invalid_encoding+=int(((a>NODATA)|( (a>SCALE)&(a!=NODATA))).sum())
                n+=a.size
                valid_total+=int(good.sum())
                if good.any():mean_sum+=float(a[good].astype(np.float64).sum())
                counts.append(float(good.mean()))
        frac=valid_total/max(n,1)
        sample_rows.append({
            "site":site,"candidate_kind":kind,"path":str(path),
            "sampled_pixels":n,"sample_valid_pixels":valid_total,
            "sample_valid_fraction":frac,
            "sample_mean_probability":mean_sum/valid_total/SCALE if valid_total else np.nan,
            "sample_invalid_encoding_pixels":invalid_encoding,
            "approx_valid_ha_NOT_EXACT":area_ha*frac,
            "sample_window_valid_fraction_min":min(counts),
            "sample_window_valid_fraction_max":max(counts),
        })
coverage=pd.DataFrame(sample_rows)
coverage.to_csv(VALIDATION/"02_stratified_coverage.csv",index=False)
display(coverage[["site","candidate_kind","sample_valid_fraction",
                  "sample_mean_probability","sample_window_valid_fraction_min",
                  "sample_window_valid_fraction_max","sample_invalid_encoding_pixels"]].round(4))
assert (coverage.sample_invalid_encoding_pixels==0).all(),"Out-of-range probabilities found."
print("Sample coverage is indicative only. ROI masks, orthomosaic edges and clustering")
print("mean we must NOT use the approximate area as exact valid-imagery area.")

# Select the broadest available candidate per site for **inspection only**.
# Final acceptance requires rigorous coverage analysis and source provenance.
rank={"full_mosaic":0,"pre_ROI_backup":1,"current":2}
coverage["priority"]=coverage.candidate_kind.map(rank)
selected=(coverage.sort_values(["site","sample_valid_fraction","priority"],
                              ascending=[True,False,True])
                  .drop_duplicates("site").copy())
selected["audit_decision"]="INSPECT_REQUIRED"
selected["reason"]="Coverage and full-mosaic provenance not yet independently certified"
selected.to_csv(VALIDATION/"02_candidate_selection_for_review.csv",index=False)
print("\nRECOMMENDED CANDIDATES FOR COVERAGE REVIEW (not accepted as FULL):")
display(selected[["site","candidate_kind","sample_valid_fraction","audit_decision"]].round(4))
missing=set(SITES)-set(selected.site)
if missing:print("MISSING candidate sites:",sorted(missing))
print("DATA GATE: full-mosaic status and satellite grid alignment remain unresolved.")
print("No satellite target grids or training tables have been generated.")

## 03 — UAV RGB footprint versus U-Net prediction coverage

**Critical audit step (CPU/read-only).** On the same sampled spatial locations, compare valid RGB coverage to valid probability coverage. This avoids concluding that sparse predictions are incomplete when much of the orthomosaic is background. The RGB proxy excludes exact black, white and blue pixels; its reliability must be visually checked, and Luki's defective fourth-band raster mask must not be used to discard actual valid RGB.

In [ ]:
# BLOCK 03 — Compare U-Net probability coverage against valid UAV RGB (CPU/read-only).
# Same 12x12 stratified locations as Block 02, larger 256-pixel windows.
# No requirement to load the full very large orthomosaics into memory.
from rasterio.windows import Window
import numpy as np,pandas as pd,rasterio
RGB_SAMPLE_PX=256
RGB_GRID_N=12
res=[]
for site in SITES:
    ip=RGB[site]
    if not ip.exists():continue
    opts=[("current",CURRENT[site])]
    if site in BACKUP:opts.append(("pre_ROI_backup",BACKUP[site]))
    if site=="Luki":opts.append(("full_mosaic",FULL_LUKI))
    opts=[(name,p) for name,p in opts if p.exists()]
    if not opts:continue
    with rasterio.open(ip) as src:
        assert src.count>=3,f"{site}: expected RGB bands"
        for variant,pp in opts:
            counts={x:0 for x in ("sample_pixels","rgb_nonbackground","rgb_dataset_mask",
                                  "prob_valid","both_nonbackground_and_prob","rgb_without_prob",
                                  "prob_without_rgb","nonbg_black","nonbg_white","nonbg_blue")}
            with rasterio.open(pp) as prob:
                assert prob.width==src.width and prob.height==src.height
                for yi in range(RGB_GRID_N):
                    for xi in range(RGB_GRID_N):
                        rr=max(0,min(int((yi+.5)/RGB_GRID_N*src.height)-RGB_SAMPLE_PX//2,
                                     src.height-RGB_SAMPLE_PX))
                        cc=max(0,min(int((xi+.5)/RGB_GRID_N*src.width)-RGB_SAMPLE_PX//2,
                                     src.width-RGB_SAMPLE_PX))
                        win=Window(cc,rr,min(src.width,RGB_SAMPLE_PX),min(src.height,RGB_SAMPLE_PX))
                        rgb=src.read([1,2,3],window=win)
                        pred=prob.read(1,window=win)
                        black=np.all(rgb==0,axis=0)
                        white=np.all(rgb==255,axis=0)
                        blue=(rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255)
                        # Conservative data-based proxy. Black/white/blue can be real
                        # reflectance values, so compare with raster mask and visually QA.
                        image_ok=~(black|white|blue)
                        declared_ok=src.dataset_mask(window=win)>0
                        pred_ok=(pred!=65535)&(pred<=10000)
                        counts["sample_pixels"]+=int(pred.size)
                        counts["rgb_nonbackground"]+=int(image_ok.sum())
                        counts["rgb_dataset_mask"]+=int(declared_ok.sum())
                        counts["prob_valid"]+=int(pred_ok.sum())
                        counts["both_nonbackground_and_prob"]+=int((image_ok&pred_ok).sum())
                        counts["rgb_without_prob"]+=int((image_ok&~pred_ok).sum())
                        counts["prob_without_rgb"]+=int((~image_ok&pred_ok).sum())
                        counts["nonbg_black"]+=int(black.sum())
                        counts["nonbg_white"]+=int(white.sum())
                        counts["nonbg_blue"]+=int(blue.sum())
            c=counts;n=max(c["sample_pixels"],1)
            valid_rgb=max(c["rgb_nonbackground"],1)
            res.append(dict(site=site,variant=variant,**counts,
                rgb_nonbackground_fraction=c["rgb_nonbackground"]/n,
                rgb_dataset_mask_fraction=c["rgb_dataset_mask"]/n,
                probability_fraction_of_bbox=c["prob_valid"]/n,
                probability_coverage_of_valid_RGB=c["both_nonbackground_and_prob"]/valid_rgb,
                valid_RGB_missing_prediction_fraction=c["rgb_without_prob"]/valid_rgb,
                prediction_outside_RGB_proxy_fraction=c["prob_without_rgb"]/max(c["prob_valid"],1)))
q=pd.DataFrame(res)
q.to_csv(VALIDATION/"03_RGB_vs_probability_coverage.csv",index=False)
view=["site","variant","rgb_nonbackground_fraction","rgb_dataset_mask_fraction",
      "probability_fraction_of_bbox","probability_coverage_of_valid_RGB",
      "valid_RGB_missing_prediction_fraction","prediction_outside_RGB_proxy_fraction"]
display(q[view].round(4))
if len(q):
    for row in q.itertuples():
        if row.probability_coverage_of_valid_RGB<0.90:
            print(f"CHECK {row.site} [{row.variant}]: "
                  f"only {row.probability_coverage_of_valid_RGB:.1%} sampled valid RGB covered")
print("IMPORTANT: This is a STRATIFIED SAMPLE, not an exact area audit.")
print("RGB validity is a heuristic (exact black, white and blue excluded).")
print("Luki dataset_mask may be zero because its fourth band is undefined/nodata=255.")
print("NEXT GATE: inspect the results and a visual RGB + prediction overlay before deciding on further inference.")


## 04 — Five-site FULL production map acceptance QA

**Production complete (11 Oct 2026):** 5/5 full-resolution UAV mosaics have been inferred using the frozen checkpoint; the existing ROI-limited products remain unchanged. The full outputs are the authoritative reference inputs for satellite target generation. **Run Block 04 after Block 00** to independently check the new products' georeferencing, schema, production provenance and pixel-level probability coverage against the exact RGB-background validity convention used in NB04 Block 10. Sampling is deliberately bounded (12 × 12 × 256² pixels per site). A PASS does not replace complete spatial QA or visualization. Luki remains in EPSG:32733; other sites EPSG:32635.

In [ ]:
# BLOCK 04 — Verify authoritative FULL-mosaic products (CPU-only, read-only).
# Fast stratified paired RGB/probability pixel QA; no model execution.
import json
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from IPython.display import display

FULL = {site: MAPS/f"{site}_musanga_probability_uint16_FULL.tif" for site in SITES}
REPORT = VALIDATION/"04_full_mosaic_production_QA.csv"
EXPECTED_MODEL = "final_unet_resnet18_2a5a0fb4d0de_ep04.pt"
rows=[]
GRID4=12
SIZE4=256
for site in SITES:
    rgb_path=RGB[site];prob_path=FULL[site]
    rec=dict(site=site,full_map=str(prob_path),full_exists=prob_path.exists())
    if not rgb_path.exists() or not prob_path.exists():
        rec["status"]="MISSING_RGB_OR_FULL_MAP";rows.append(rec);continue
    with rasterio.open(rgb_path) as src,rasterio.open(prob_path) as pred:
        same_grid=(src.width==pred.width and src.height==pred.height and
                   src.crs==pred.crs and src.transform==pred.transform)
        schema=(pred.count==1 and pred.dtypes[0]=="uint16" and pred.nodata==65535)
        tags=pred.tags()
        tag_ok=(tags.get("COVERAGE")=="FULL_VALID_RGB" and
                tags.get("COMPLETE")=="YES" and
                tags.get("MODEL_CHECKPOINT")==EXPECTED_MODEL and
                tags.get("MUSANGA_SCALE")=="10000" and
                bool(tags.get("SPEC_FINGERPRINT")))
        rec.update(crs=str(src.crs),same_grid=same_grid,schema_ok=schema,
                   provenance_ok=tag_ok,model=tags.get("MODEL_CHECKPOINT"),
                   fingerprint=tags.get("SPEC_FINGERPRINT"),
                   full_file_GB=round(prob_path.stat().st_size/1e9,3))
        if not (same_grid and schema and tag_ok):
            rec["status"]="METADATA_FAIL";rows.append(rec);continue
        counts=dict(n=0,rgb_valid=0,prob_valid=0,both=0,missing=0,
                    excess=0,illegal=0,special_dataset_valid=0)
        for iy in range(GRID4):
            for ix in range(GRID4):
                h=min(SIZE4,src.height);w=min(SIZE4,src.width)
                rr=max(0,min(int((iy+.5)*src.height/GRID4)-h//2,src.height-h))
                cc=max(0,min(int((ix+.5)*src.width/GRID4)-w//2,src.width-w))
                win=Window(cc,rr,w,h)
                rgb=src.read([1,2,3],window=win)
                black=np.all(rgb==0,axis=0)
                white=np.all(rgb==255,axis=0)
                blue=(rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255)
                valid_rgb=~(black|white|blue)
                p=pred.read(1,window=win)
                valid_p=(p<=10000)
                counts["n"]+=p.size
                counts["rgb_valid"]+=int(valid_rgb.sum())
                counts["prob_valid"]+=int(valid_p.sum())
                counts["both"]+=int((valid_rgb&valid_p).sum())
                counts["missing"]+=int((valid_rgb&~valid_p).sum())
                counts["excess"]+=int((~valid_rgb&valid_p).sum())
                counts["illegal"]+=int(((p>10000)&(p!=65535)).sum())
                counts["special_dataset_valid"]+=int((src.dataset_mask(window=win)>0).sum())
        rec.update(rgb_valid_sample_fraction=counts["rgb_valid"]/counts["n"],
                   probability_valid_sample_fraction=counts["prob_valid"]/counts["n"],
                   probability_coverage_given_RGB=counts["both"]/max(counts["rgb_valid"],1),
                   probability_without_RGB_fraction=counts["excess"]/max(counts["n"],1),
                   missing_RGB_pixels=counts["missing"],
                   illegal_prediction_pixels=counts["illegal"],
                   dataset_mask_sample_fraction=counts["special_dataset_valid"]/counts["n"])
        rec["status"]=("PASS_SAMPLED" if counts["missing"]==0 and counts["excess"]==0
                       and counts["illegal"]==0 else "CHECK_SAMPLE_MISMATCH")
        rows.append(rec)
df_full=pd.DataFrame(rows)
df_full.to_csv(REPORT,index=False)
show=["site","crs","full_file_GB","rgb_valid_sample_fraction",
      "probability_valid_sample_fraction","probability_coverage_given_RGB",
      "missing_RGB_pixels","illegal_prediction_pixels","status"]
display(df_full[[x for x in show if x in df_full]].round(5))
print("FULL-MOSAIC PRODUCTS:",int(df_full.full_exists.sum()),"/",len(SITES))
print("Sample status counts:",df_full.status.value_counts().to_dict())
print("Source rasters not modified. This verifies sampled equality, not yet exact whole-raster equality.")
print("NEXT: determine Sentinel-2 acquisition dates, exact pixel grids, and embedding native transforms.")


## 05 — Scientific design and modelling contract

### UAV reference target
- Create a 5 m intermediate aggregation grid per site, then align satellite-scale targets to a verified Sentinel-2 projection and pixel origin from *full-coverage* UAV maps, with pixel area weighting where necessary.
- Maintain **valid UAV observation fraction** and **mean Musanga probability** as separate bands. Derive cover-fraction alternatives based on calibrated or fixed U-Net probability thresholds; never treat unmapped pixels as zero.
- Consider crown labels / Luki inventory for independent UAV-prediction calibration and audit. Thresholds must not be optimized using held-out satellite sites.
- Require a minimum observed high-resolution area per satellite grid cell (initial proposal ≥90%; sensitivity: 70%, 80%, 95%).
- Use strictly aligned support grids at 10, 30, 50, 75 and 100 m, all integer multiples of 5 m. Do not interpolate a 10 m satellite measurement into novel 10 m independent observations.

### Satellite feature families (independently ablated)
- **S00** geographic/climatological null baseline only; avoid site identifiers as predictors in site-held-out transfer.
- **S01** Sentinel-2 SR: B2/B3/B4/B5/B6/B7/B8/B8A/B11/B12 and NDVI, EVI, NDRE, NBR, NDMI; use correct resampling of native 20 m bands.
- **S02** Sentinel-2 quality-filtered seasonal percentiles and cloud-free acquisition counts, matched to UAV acquisition epoch.
- **S03** Sentinel-2 monthly/quarterly trajectories, phenological amplitudes and long-term trend; explicitly check missing periods and date leakage.
- **E01** Earth-observation embeddings (e.g. annual Sentinel-2-based embeddings), with provenance, source year and spatial support recorded.
- **F01** spectral + temporal; **F02** embeddings + spectral; **F03** all combined.
- Optional later: Sentinel-1, terrain, forest disturbance history. Keep separate so contribution is identifiable.

### Resolution experiment
Model 10, 30, 50, 75, 100 m *support* with the same site holdouts and comparable coverage gates. Distinguish (i) physical aggregation of UAV fraction, (ii) satellite native spatial resolution, (iii) smoothing/resampling. A 10 m multispectral feature stack contains native 20 m bands and must be labelled accordingly.

### Validation design
**Primary:** five leave-one-site-out folds, training/tuning on four sites; avoid using Luki for any model choice in its held-out fold.
**Secondary:** buffered spatial-block cross-validation within the training sites, blocks substantially larger than the 100 m response grid (test 500 m / 1 km based on autocorrelation).
**No leakage:** tune preprocessing, feature selection, threshold calibration and hyperparameters inside each training fold. Sample weights should avoid dominance by large mosaics and spatially autocorrelated pixels.

### Algorithms
Reference baselines: constant mean, ridge/elastic net and random forest. Main models: HistGradientBoosting or LightGBM plus a regularized embedding model. Optional spatial CNN only after tabular transfer baseline is established. Predict bounded fractions via suitable link/loss or calibration.

### Metrics and science
Primary: site-held-out MAE and RMSE (fraction units), calibration bias and R² relative to within-site variance. Secondary: Spearman ranking, occurrence detection AUC/AP where prevalence is relevant, abundance-stratified error, quality-/coverage-stratified error and bootstrap confidence intervals by spatial block. Report site-by-site and macro averages; do not use random pixel-split scores as external transfer estimates.

### Expected outputs
- Reproducible coverage-checked 5 m UAV reference grids, plus aggregate target grids at all five scales.
- Five-site harmonized spectral/temporal/embedding feature tables and provenance.
- Predeclared experiment matrix and leak-proof fold manifests.
- LO(S)O benchmark, feature ablations, support-scale curves, SHAP/permutation-importance checks on held-out data.
- Final full-predictor model and uncertainty rasters, then ecological tests against disturbance/succession.

## 06 — Prespecified experiment registry

**Scaffold only.** Register resolution × feature-family × algorithm combinations. Do not fit models before reference coverage and satellite grid alignment have passed QA.

In [ ]:
# BLOCK 06 — Prespecified experiment registry (scaffold; no model fitting)
resolutions=[10,30,50,75,100]
feature_families=["S01_S2_static","S02_S2_seasonal","S03_S2_trajectory",
                  "E01_embeddings","F01_spectral_temporal","F02_embedding_spectral","F03_all"]
algorithms=["ridge","random_forest","hist_gradient_boosting"]
experiments=pd.DataFrame(
    [(r,f,a) for r in resolutions for f in feature_families for a in algorithms],
    columns=["resolution_m","feature_family","algorithm"]
)
experiments["status"]="NOT_RUN"
experiments.to_csv(OUT/"experiment_registry.csv",index=False)
print("Registered experiments:",len(experiments))
print("× 5 independent site holdouts; not fitted until data QA passes.")
display(experiments.head(12))